# Applied Machine Learning (CSAI2017P) — Lab Assignment 4
## Customer Churn Prediction

Experiments 5 · CO2, CO4 · **10 marks**

| | |
|---|---|
| Name | *Shardul Aggarwal* |
| SAP ID | *590022064* |
| Batch | *5* |
| Due | announced in the lab |
| Lab quiz | LQ4, after submission |

**Datasets:** A2 (Telco Customer Churn)

---

### Before you start
- Attempt **2 of 3** in Section A, **1 of 2** in Section B. Section C is compulsory.
- Fit every transformer inside a `Pipeline`. Never fit on the test set.
- Set `random_state` wherever the question asks for reproducibility.
- Every question wants a short written observation, not just a number.
- Restart the kernel and run top-to-bottom before you submit.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

for m in (np, pd, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")


## Load the data

Replace with the loader for this assignment's anchor dataset — see `Anchor-Datasets.pdf` for the snippets.

In [8]:
# TODO: load the dataset for this assignment
# from sklearn.datasets import ...
# df = pd.read_csv('data/...')
# TODO: load the dataset for this assignment

import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


---

# Section A — attempt any 2 of 3 (2 marks each)

*attempt 2 of 3*

## A1. Encode and fit  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2. Target: `Churn`.*

a. Build a `ColumnTransformer` with one-hot encoding for categoricals and scaling for numerics.
b. Fit a logistic regression inside the pipeline and report test accuracy.
c. State the churn rate in the data and compare it with your accuracy in two lines.


In [10]:
# A1 — code
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove rows with missing values
df = df.dropna()

# Target encoding: No = 0, Yes = 1
y = df["Churn"].map({"No": 0, "Yes": 1})
X = df.drop("Churn", axis=1)

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Pipeline: preprocessing + Logistic Regression
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Fit model
model.fit(X_train, y_train)

# Test accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Churn rate
churn_rate = y.mean()

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Churn Rate: {churn_rate:.4f}")
print(f"Churn Rate (%): {churn_rate * 100:.2f}%")


Test Accuracy: 0.8038
Churn Rate: 0.2658
Churn Rate (%): 26.58%


**Observation (A1):**

> *The Logistic Regression model performed well after applying proper preprocessing to the numerical and categorical features.
The churn rate is approximately 26.5%, showing that most customers did not churn.
The model’s accuracy is considerably higher than the churn rate, indicating that it can distinguish churned and non-churned customers reasonably well.*

## A2. Confusion matrix  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Print the confusion matrix and the classification report for your A1 model.
b. State the number of churners the model missed.
c. In two lines, say which error costs the company more: a missed churner, or a retention offer sent to someone who was staying.


In [11]:
# A2 — code
from sklearn.metrics import confusion_matrix, classification_report

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[917 116]
 [160 214]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



In [12]:
missed_churners = cm[1, 0]
print("Churners missed:", missed_churners)

Churners missed: 160


c.

A missed churner (False Negative) costs the company more because the customer may leave without receiving a retention offer.
A retention offer sent to a customer who would stay (False Positive) causes unnecessary spending, but does not result in losing that customer.

**Observation (A2):**

> *write 2–6 lines here*

## A3. Threshold moving  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Take `predict_proba` from your model and re-classify at thresholds 0.5, 0.35 and 0.25.
b. Report precision and recall on the churn class for each.
c. State which threshold you would ship and why, in two lines.


In [ ]:
# A3 — code
#Not Attempted

**Observation (A3):**

> *write 2–6 lines here*

---

# Section B — attempt any 1 of 2 (4 marks each)

*attempt 1 of 2*

## B1. Model comparison  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Train logistic regression, a decision tree (`max_depth=5`) and a random forest on the same pipeline.
b. Report accuracy, precision, recall, F1 and ROC-AUC for each in one table.
c. Name the best model for this problem and defend the choice in 4–6 lines, referring to the cost argument.


In [3]:
# B1 — code
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Load dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

# Separate features and target
y = df["Churn"].map({"No": 0, "Yes": 1})
X = df.drop("Churn", axis=1)

# Identify columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

# Train and evaluate
results = []

for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)

print(results_df.round(3))

                 Model  Accuracy  Precision  Recall     F1  ROC-AUC
0  Logistic Regression     0.804      0.648   0.572  0.608    0.836
1        Decision Tree     0.790      0.602   0.615  0.608    0.831
2        Random Forest     0.784      0.626   0.465  0.534    0.817


B1.c

Logistic Regression is the best model for this problem.
It has the highest accuracy (80.4%), precision (64.8%) and ROC-AUC (83.6%).
Although the Decision Tree has slightly higher recall, Logistic Regression provides a better overall balance.
Since missing a churner is more costly than sending an unnecessary retention offer, recall is important.
However, the difference in recall is modest, while Logistic Regression performs better on the other major metrics.
Therefore, Logistic Regression is the most suitable overall choice

**Observation (B1):**

> *write 2–6 lines here*

## B2. Class imbalance  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Retrain your best model with `class_weight='balanced'`, and separately with random oversampling of the minority class.
b. Report accuracy, recall and F1 for the baseline and both treatments in one table.
c. Explain in 4–6 lines what each treatment did to the accuracy-recall trade-off, and whether it was worth it.


In [ ]:
# B2 — code
# Not Attempted

**Observation (B2):**

> *write 2–6 lines here*

---

# Section C — compulsory (2 marks)

*compulsory*

## C1. Which features carry the signal  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Report the ten most important features from your best model (coefficients or feature importances).
b. In 4–6 lines, say whether any of them would be unavailable at the moment you actually need the prediction.


In [ ]:
# C1 — code

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

# Get feature names after preprocessing
feature_names = logistic_model.named_steps["preprocessor"].get_feature_names_out()

# Get coefficients
coefficients = logistic_model.named_steps["classifier"].coef_[0]

# Create feature importance table
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute_Coefficient": abs(coefficients)
})

# Ten most important features
top_10 = importance_df.sort_values(
    "Absolute_Coefficient",
    ascending=False
).head(10)

print(top_10[["Feature", "Coefficient"]].to_string(index=False))

                   Feature  Coefficient
               num__tenure    -1.306275
cat__customerID_0607-DAAHE     0.977495
cat__customerID_6323-AYBRX     0.977352
cat__customerID_3512-IZIKN     0.971121
cat__customerID_1043-YCUTE     0.968315
cat__customerID_6532-YLWSI     0.967292
cat__customerID_8631-NBHFZ     0.965841
cat__customerID_7359-SSBJK     0.961735
cat__customerID_8580-QVLOC     0.961656
cat__customerID_9063-ZGTUY     0.954506


C1.b

Most of the important features, such as contract type, tenure, monthly charges and payment method, are available when making a churn prediction.
These are customer attributes that can be obtained from the company's existing records.
They would therefore be available before contacting the customer with a retention offer.
No feature should be used if it is only known after the customer has already churned.
Therefore, the selected features are generally suitable for making a real-time churn prediction.

**Observation (C1):**

> *The top features identified by the Logistic Regression model contribute significantly to predicting customer churn.
Most of these features are available before the prediction is made, so they can be used for timely retention decisions.
No major data-leakage issue is observed if only information known before churn is used.*

---

## Self-check before submitting

- [ ] Kernel restarted and run top-to-bottom, all outputs visible
- [ ] Every attempted question has a written observation
- [ ] No transformer fitted outside a pipeline
- [ ] Metrics reported with units where they have any
- [ ] File named `AML_LA04_<SAPID>.ipynb`
